# Validação de Precificação de Títulos Públicos Federais (TPFs)

Este caderno documenta a precificação analítica, a decomposição fluxo a fluxo e a validação empírica dos Títulos Públicos Federais (TPFs) negociados no mercado brasileiro, seguindo estritamente as convenções matemáticas, financeiras e de truncamento padronizadas pela Associação Brasileira das Entidades dos Mercados Financeiro e de Capitais (ANBIMA) e pela Secretaria do Tesouro Nacional (STN).

O objetivo central consiste em demonstrar:
1. A extração e modelagem individualizada dos fluxos de caixa de cada família de títulos públicos (**LTN**, **NTN-F**, **LFT**, **NTN-B**, **NTN-B Principal** e **NTN-C**).
2. O cruzamento automatizado com a totalidade dos **50 títulos públicos vigentes** na data de referência de **30/10/2026** divulgados no arquivo oficial da ANBIMA.
3. A conformidade exata das regras de liquidação financeira no primeiro dia útil subsequente (`following`) para vencimentos nominais não úteis.


In [ ]:
import datetime
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from bz_pricing.core.calendars import get_calendar
from bz_pricing.core.curves import FlatCurve, SimpleCurve
from bz_pricing.instruments.tpf import LTN, NTNF, LFT, NTNB, NTNBPrincipal, NTNC

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.figsize'] = (12, 6)

cal = get_calendar('B3')
base_date = datetime.date(2026, 10, 30)
print(f"Data-base de análise: {base_date.strftime('%d/%m/%Y')}")


## 1. Leitura dos Insumos Oficiais da ANBIMA e Tesouro Nacional

Carregam-se os dados consolidados do arquivo oficial da ANBIMA (`ANBIMA_MS_20261030.csv`) e o histórico de pagamentos de cupons do Tesouro Direto (`Pagamento_Cupom_Juros_TD.csv`).


In [ ]:
# Leitura do arquivo oficial ANBIMA (30/10/2026)
df_anbima = pd.read_csv('../dados/b3/ANBIMA_MS_20261030.csv', sep='@', encoding='latin1')
print(f"Total de registros na base ANBIMA: {len(df_anbima)}")
print(df_anbima['Titulo'].value_counts())

# Leitura do histórico de pagamento de cupons do Tesouro Direto
df_cupons_td = pd.read_csv('../dados/b3/Pagamento_Cupom_Juros_TD.csv', sep=';')
print(f"\nTotal de registros de cupons liquidados no TD: {len(df_cupons_td)}")
print(df_cupons_td.head(3))


## 2. Decomposição Fluxo a Fluxo por Família de Instrumento

Demonstra-se a extração da tabela detalhada de fluxos de caixa através do método `cashflow_table` e do parâmetro `return_cashflows=True` implementados nas classes de instrumentos públicos.


In [ ]:
# 1. NTN-F 2029 (Cupom semestral de 10% a.a.)
tx_ntnf = 0.149501
curva_ntnf = FlatCurve(tx_ntnf)
ntnf = NTNF(maturity_date='2029-01-01', coupon_rate=0.10)
pu_ntnf, df_ntnf = ntnf.present_value(base_date, curva_ntnf, return_cashflows=True)

print(f"NTN-F 2029: PU Calculado = R$ {pu_ntnf:.6f}")
display(df_ntnf)

# 2. NTN-B 2027 (Taxa real de 4,6629% a.a. e VNA = 4771.575461)
tx_ntnb = 0.046629
vna_ntnb = 4771.575461
curva_ntnb = FlatCurve(tx_ntnb)
ntnb = NTNB(maturity_date='2027-05-15', coupon_rate=0.06)
pu_ntnb, df_ntnb = ntnb.present_value(base_date, curva_ntnb, vna=vna_ntnb, return_cashflows=True)

print(f"\nNTN-B 2027: PU Calculado = R$ {pu_ntnb:.6f} | Cotação = {ntnb.cotacao(base_date, curva_ntnb):.4f}%")
display(df_ntnb)


## 3. Cruzamento em Lote: 50 Títulos Públicos Vigentes em 30/10/2026

Executa-se a precificação individualizada de cada um dos 50 ativos vigentes confrontando o resultado com o PU oficial divulgado pela ANBIMA.


In [ ]:
vna_lft = 20201.049997
vna_ntnb = 4771.575461
vna_ntnc = 8321.942046

results = []
for idx, row in df_anbima.iterrows():
    tit = str(row['Titulo']).strip()
    dt_vcto_str = str(int(row['Data Vencimento']))
    dt_vcto = datetime.date(int(dt_vcto_str[:4]), int(dt_vcto_str[4:6]), int(dt_vcto_str[6:8]))
    
    tx_str = str(row['Tx. Indicativas']).replace('.', '').replace(',', '.')
    tx = float(tx_str) / 100.0
    
    pu_ofic_str = str(row['PU']).replace('.', '').replace(',', '.')
    pu_ofic = float(pu_ofic_str)
    
    curva = FlatCurve(tx)
    
    if tit == 'LTN':
        bond = LTN(dt_vcto)
        pu_calc, cf_df = bond.present_value(base_date, curva, return_cashflows=True)
    elif tit == 'NTN-F':
        bond = NTNF(dt_vcto, coupon_rate=0.10)
        pu_calc, cf_df = bond.present_value(base_date, curva, return_cashflows=True)
    elif tit == 'LFT':
        bond = LFT(dt_vcto)
        pu_calc, cf_df = bond.present_value(base_date, curva, vna=vna_lft, return_cashflows=True)
    elif tit == 'NTN-B':
        bond = NTNB(dt_vcto, coupon_rate=0.06)
        pu_calc, cf_df = bond.present_value(base_date, curva, vna=vna_ntnb, return_cashflows=True)
    elif tit == 'NTN-C':
        bond = NTNC(dt_vcto, coupon_rate=0.06)
        pu_calc, cf_df = bond.present_value(base_date, curva, vna=vna_ntnc, return_cashflows=True)
    else:
        continue
        
    diff = pu_calc - pu_ofic
    dur = bond.duration(base_date, curva)
    
    results.append({
        'Titulo': tit,
        'Vencimento': dt_vcto,
        'Tx_Indicativa': tx * 100.0,
        'PU_Oficial': pu_ofic,
        'PU_Calculado': pu_calc,
        'Divergencia': diff,
        'Duration_DU': dur,
        'Qtd_Fluxos': len(cf_df)
    })

res_df = pd.DataFrame(results)
print(f"Total de títulos avaliados: {len(res_df)}")
print(f"Divergência Máxima Absoluta: R$ {res_df['Divergencia'].abs().max():.6f}")
print(f"Casos com |Divergência| >= R$ 0,01: {(res_df['Divergencia'].abs() >= 0.01).sum()}")

# Tabela resumo por família de títulos
resumo_familia = res_df.groupby('Titulo').agg(
    Qtd_Titulos=('PU_Oficial', 'count'),
    Vencimento_Min=('Vencimento', 'min'),
    Vencimento_Max=('Vencimento', 'max'),
    Max_Abs_Diff=('Divergencia', lambda x: x.abs().max())
).reset_index()

display(resumo_familia)


## 4. Análise Gráfica e Estrutura Temporal dos Fluxos


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico 1: Dispersão PU Calculado vs PU Oficial
for tit, grp in res_df.groupby('Titulo'):
    axes[0].scatter(grp['PU_Oficial'], grp['PU_Calculado'], label=tit, s=50, alpha=0.85)

min_pu = res_df['PU_Oficial'].min() * 0.95
max_pu = res_df['PU_Oficial'].max() * 1.05
axes[0].plot([min_pu, max_pu], [min_pu, max_pu], 'r--', alpha=0.6, label='Linha de Aderência Exata (45°)')
axes[0].set_title('Confronto de Preço Unitário: Calculado vs Oficial ANBIMA', fontsize=12, fontweight='bold')
axes[0].set_xlabel('PU Oficial ANBIMA (R$)', fontsize=11)
axes[0].set_ylabel('PU Calculado pelo Modelo (R$)', fontsize=11)
axes[0].legend(loc='upper left')

# Gráfico 2: Estrutura Temporal de Fluxos Descontados para NTN-F 2037
ntnf_longa = NTNF('2037-01-01', coupon_rate=0.10)
tx_ntnf_longa = df_anbima[(df_anbima['Titulo'] == 'NTN-F') & (df_anbima['Data Vencimento'] == 20370101)]['Tx. Indicativas'].iloc[0]
tx_longa = float(str(tx_ntnf_longa).replace(',', '.')) / 100.0
pu_l, df_fl_longa = ntnf_longa.present_value(base_date, FlatCurve(tx_longa), return_cashflows=True)

df_fl_longa['ano'] = pd.to_datetime(df_fl_longa['data_fluxo']).dt.year
axes[1].bar(df_fl_longa['du'], df_fl_longa['valor_presente'], width=50, color='darkblue', alpha=0.7, label='Valor Presente do Fluxo (R$)')
axes[1].plot(df_fl_longa['du'], df_fl_longa['fluxo_nominal'], 'ro-', markersize=4, alpha=0.6, label='Fluxo Nominal Contratual (R$)')
axes[1].set_title('Perfil Temporal de Fluxos: NTN-F 2037', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Prazo até a Liquidação (Dias Úteis - DU)', fontsize=11)
axes[1].set_ylabel('Valor (R$)', fontsize=11)
axes[1].legend(loc='upper right')

plt.tight_layout()
plt.show()


## 5. Referências Normativas e Bibliográficas

- **ANBIMA – Associação Brasileira das Entidades dos Mercados Financeiro e de Capitais.** *Metodologia de Precificação de Títulos Públicos Federais e Estrutura a Termo das Taxas de Juros (ETTJ)*. Rio de Janeiro: ANBIMA. Disponível em: <https://www.anbima.com.br/pt_br/informar/precos-e-indices/precos/titulos-publicos.htm>.
- **Secretaria do Tesouro Nacional (STN).** *Regras e Fórmulas de Cálculo dos Títulos Públicos Federais no Mercado Secundário*. Brasília: Ministério da Fazenda. Disponível em: <https://www.tesourodireto.com.br/titulos/historico-de-precos-e-taxas.htm>.
- **Banco Central do Brasil (BCB).** *Sistema Especial de Liquidação e Custódia (Selic): Manuais de Operações e Liquidação Financeira*. Brasília: BCB. Disponível em: <https://www.bcb.gov.br/>.
